In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
train_df = pd.read_parquet("../data/processed/casp9_features.parquet")
val_df = pd.read_parquet("../data/processed/validation_features.parquet")
test_df = pd.read_parquet("../data/processed/testing_features.parquet")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(3337092, 51)
(47602, 51)
(24466, 51)


In [3]:
feature_cols = [c for c in train_df.columns if c.startswith("Evo") or c.startswith("AA_")]

X_train = train_df[feature_cols]
y_train = train_df["CentroidDist"]

X_val = val_df[feature_cols]
y_val = val_df["CentroidDist"]

X_test = test_df[feature_cols]
y_test = test_df["CentroidDist"]

Convert to NumPy (for GPU)

In [4]:
X_train_np = X_train.values
y_train_np = y_train.values

X_val_np = X_val.values
y_val_np = y_val.values

X_test_np = X_test.values
y_test_np = y_test.values

In [5]:
xgb_model = XGBRegressor(

    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method="hist",   # GPU histogram algorithm
    device="cuda",        # forces GPU usage

    max_bin=256,          # reduces GPU memory usage

    random_state=42,
    verbosity=1
)

In [6]:
print("Training XGBoost on GPU...")

xgb_model.fit(
    X_train_np,
    y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    verbose=True
)

Training XGBoost on GPU...
[0]	validation_0-rmse:839.83864
[1]	validation_0-rmse:836.42188
[2]	validation_0-rmse:832.80710
[3]	validation_0-rmse:829.66683
[4]	validation_0-rmse:827.05514
[5]	validation_0-rmse:824.76523
[6]	validation_0-rmse:822.58532
[7]	validation_0-rmse:820.39322
[8]	validation_0-rmse:818.39257
[9]	validation_0-rmse:816.74716
[10]	validation_0-rmse:815.16848
[11]	validation_0-rmse:813.58891
[12]	validation_0-rmse:812.41991
[13]	validation_0-rmse:811.44372
[14]	validation_0-rmse:811.08313
[15]	validation_0-rmse:810.05999
[16]	validation_0-rmse:809.14464
[17]	validation_0-rmse:808.25919
[18]	validation_0-rmse:807.67825
[19]	validation_0-rmse:807.10601
[20]	validation_0-rmse:806.67372
[21]	validation_0-rmse:806.23737
[22]	validation_0-rmse:805.85137
[23]	validation_0-rmse:805.44191
[24]	validation_0-rmse:805.05763
[25]	validation_0-rmse:804.77815
[26]	validation_0-rmse:804.53940
[27]	validation_0-rmse:804.31679
[28]	validation_0-rmse:804.03338
[29]	validation_0-rmse:803

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [7]:
preds = xgb_model.predict(X_test_np)

d:\ProteinPrediction\MainFiles\GPU_VENV\lib\site-packages\xgboost\core.py:751: UserWarning: [14:23:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [8]:
rmse = np.sqrt(mean_squared_error(y_test_np, preds))
mae = mean_absolute_error(y_test_np, preds)
r2 = r2_score(y_test_np, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 914.2172198115719
MAE: 630.8269653320312
R2: 0.02542293071746826
